# EQCCT-torch → EQViT magnitude: realistic pick-error training

**EQViT-torch Phase 2** — reproducible, event-disjoint, latency-aware workflow.

> Research prototype: validate locally before operational EEW use.

## Why predicted P matters

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from torch.utils.data import DataLoader
from eqvit_torch import *
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
H5=Path('/path/to/TXED_20231111.h5'); IDS=Path('/path/to/ID_20231111.npy')
SEED=2026; np.random.seed(SEED); torch.manual_seed(SEED)
print('device:',DEVICE)

In [ ]:
from eqvit_torch.eqcct_adapter import load_pick_cache
# Generate `eqcct_picks.npz` with your installed EQCCT-torch using arrays: ids, p_samples, p_prob.
# This loose cache boundary deliberately survives EQCCT API changes.
pick_cache=load_pick_cache('eqcct_picks.npz')
print('cached picks:',len(pick_cache))

In [ ]:
import h5py
err=[]
with h5py.File(H5,'r') as f:
 for wid,(p,q) in list(pick_cache.items()):
  if wid in f and 'p_arrival_sample' in f[wid].attrs: err.append((p-int(f[wid].attrs['p_arrival_sample']))/100.)
err=np.asarray(err); print('P MAE(s)=',np.mean(abs(err)),'sigma=',np.std(err)); plt.hist(err,bins=80); plt.xlabel('EQCCT P − manual P (s)'); plt.show()

In [ ]:
tr,va,te=split_ids_by_event(IDS,seed=SEED)
tr=[x for x in tr if x in pick_cache]; va=[x for x in va if x in pick_cache]; te=[x for x in te if x in pick_cache]
train_ds=TXEDDataset(H5,IDS,tr,pick_cache=pick_cache); val_ds=TXEDDataset(H5,IDS,va,pick_cache=pick_cache); test_ds=TXEDDataset(H5,IDS,te,pick_cache=pick_cache)
print(len(train_ds),len(val_ds),len(test_ds))

In [ ]:
# Compare manual-P, EQCCT-P, and deliberately jittered-P magnitude performance.
# A robust operational model should degrade gracefully as |P error| increases.
print('Recommended bins: 0–0.05, 0.05–0.1, 0.1–0.2, 0.2–0.5, >0.5 s')